In [1]:
# # 002_dataset_audit.ipynb

# 1. Imports
#       |
#       ↓
# 2. Get WaxalNLP configs
#       |
#       ↓
# 3. Filter _asr configs
#       |
#       ↓
# 4. Load ful_asr
#       |
#       ↓
# 5. Inspect features
#       |
#       ↓
# 6. Inspect samples
#       |
#       ↓
# 7. Create dataset summary

/workspace/waxal-asr-project/.venv/bin/python


In [1]:
import sys
print(sys.executable)

/workspace/waxal-asr-project/.venv/bin/python


In [18]:
# Import

import os
import pandas as pd
import numpy as np

from datasets import load_dataset, get_dataset_config_names

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

In [19]:
# Check available WaxalNLP datasets

waxal_configs = get_dataset_config_names(
    "google/WaxalNLP"
)

waxal_configs

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

['ach_asr',
 'ach_tts',
 'aka_asr',
 'amh_asr',
 'bau_tts',
 'dag_asr',
 'dga_asr',
 'ewe_asr',
 'ewe_tts',
 'fat_tts',
 'ful_asr',
 'ful_tts',
 'hau_tts',
 'ibo_tts',
 'kik_tts',
 'kpo_asr',
 'lin_asr',
 'lug_asr',
 'lug_tts',
 'luo_tts',
 'mas_asr',
 'mlg_asr',
 'nyn_asr',
 'nyn_tts',
 'orm_asr',
 'pcm_tts',
 'sid_asr',
 'sna_asr',
 'tir_asr',
 'sog_asr',
 'swa_tts',
 'twi_tts',
 'yor_tts',
 'wal_asr']

In [20]:
# Keep only ASR datasets
waxal_asr_configs = [
    x for x in waxal_configs
    if x.endswith("_asr")
]

waxal_asr_configs

['ach_asr',
 'aka_asr',
 'amh_asr',
 'dag_asr',
 'dga_asr',
 'ewe_asr',
 'ful_asr',
 'kpo_asr',
 'lin_asr',
 'lug_asr',
 'mas_asr',
 'mlg_asr',
 'nyn_asr',
 'orm_asr',
 'sid_asr',
 'sna_asr',
 'tir_asr',
 'sog_asr',
 'wal_asr']

In [3]:
# Fulfulde is a major African language and has ASR data.

fulfulde = load_dataset(
    "google/WaxalNLP",
    "ful_asr",
    # streaming=True
    # streaming=True
)

fulfulde

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/44 [00:00<?, ?it/s]

IterableDatasetDict({
    train: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 11
    })
    validation: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 2
    })
    test: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 2
    })
    unlabeled: IterableDataset({
        features: ['id', 'speaker_id', 'transcription', 'language', 'gender', 'audio'],
        num_shards: 44
    })
})

In [21]:
# Load Nigerian Common Voice
hausa = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "hausa"
)

igbo = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "igbo"
)

yoruba = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "yoruba"
)

In [8]:
# Normalize Common Voice
def normalize_common_voice(example, language):

    return {
        "audio": example["audio"],
        "text": example["sentence"],
        "language": language,
        "speaker_id": example["client_id"],
    }

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'client_id': Value('string'),
 'path': Value('string'),
 'sentence': Value('string'),
 'accent': Value('string'),
 'locale': Value('string')}

In [22]:
# Apply

# hausa = hausa.map(
#     lambda x: normalize_common_voice(x, "hau")
# )

# igbo = igbo.map(
#     lambda x: normalize_common_voice(x, "ibo")
# )

# yoruba = yoruba.map(
#     lambda x: normalize_common_voice(x, "yor")
# )

columns_to_remove = [
    "client_id",
    "path",
    "sentence",
    "accent",
    "locale"
]

hausa = hausa.remove_columns(columns_to_remove)
igbo = igbo.remove_columns(columns_to_remove)
yoruba = yoruba.remove_columns(columns_to_remove)

In [24]:
#  check
hausa["train"].features

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'client_id': Value('string'),
 'path': Value('string'),
 'sentence': Value('string'),
 'accent': Value('string'),
 'locale': Value('string'),
 'text': Value('string'),
 'language': Value('string'),
 'speaker_id': Value('string')}

In [25]:
# create

def normalize_waxal(example):

    return {
        "audio": example["audio"],
        "text": example["transcription"],
        "language": example["language"],
        "speaker_id": example["speaker_id"]
    }

In [26]:
# Apply

fulfulde = fulfulde.map(
    normalize_waxal
)

In [27]:
# check
fulfulde["train"].features

In [28]:
for name, ds in datasets.items():
    print("\n", name)
    print(ds)

In [ ]:
for name, ds in datasets.items():

    print("\n==========")
    print(name)

    sample = next(iter(ds["train"]))

    print(sample["text"] if "text" in sample else sample["transcription"])

In [ ]:
for name, ds in datasets.items():

    print(name)

    sample = next(iter(ds["train"]))

    print(sample["language"])

In [ ]:
# #  Finalise
# datasets = {
#     "hausa": hausa,
#     "igbo": igbo,
#     "yoruba": yoruba,
#     "fulfulde": fulfulde
# }

In [ ]:
#  Split before comine
#  Convert each split
from datasets import Dataset

In [ ]:
#  for training
fulfulde_train = Dataset.from_list(
    list(fulfulde["train"])
)

In [ ]:
#  for validation

fulfulde_validation = Dataset.from_list(
    list(fulfulde["validation"])
)

In [ ]:
#  Test
# This downloads/reads the data, so it may take time.

fulfulde_test = Dataset.from_list(
    list(fulfulde["test"])
)

In [ ]:
#  Split before comine
from datasets import Dataset

In [ ]:
# combine
train_sets = [
    hausa["train"],
    igbo["train"],
    yoruba["train"],
    fulfulde["train"]
]

In [22]:
# Apply

hausa = hausa.map(
    lambda x: normalize_common_voice(x, "hau")
)

igbo = igbo.map(
    lambda x: normalize_common_voice(x, "ibo")
)

yoruba = yoruba.map(
    lambda x: normalize_common_voice(x, "yor")
)

In [28]:
# Nigerian Pidgin
configs = get_dataset_config_names(
    "benjaminogbonna/nigerian_common_voice_dataset"
)

configs

['english', 'hausa', 'igbo', 'yoruba']

In [29]:
# Load Nigerian Pidgin
pidgin = load_dataset(
    "benjaminogbonna/nigerian_common_voice_dataset",
    "nigerian-pidgin"
)

ValueError: BuilderConfig 'nigerian-pidgin' not found. Available: ['english', 'hausa', 'igbo', 'yoruba']

In [ ]:
#  Finalise
datasets = {
    "hausa": hausa,
    "igbo": igbo,
    "yoruba": yoruba,
    "fulfulde": fulfulde
}